# Train and select SparseComm without held-out tuning

The actor chooses one of 55 explicit actions: NoComm, or one of all 15 directed two-edge supports, two opposite cycles, or broadcast at quota 64/128/256. This makes the sampled action exactly the graph/quota executed. NoComm incurs zero additional calls and is a legitimate learned decision. Do not force nonzero edges or claim adaptation if the policy selects one action for every question.

Observable features are citation/evidence availability, source-message lengths, answer disagreement, lexical relevance and fixed role assignments. No gold correctness, answer labels, ID hashes or accuracy history is a controller input. Gold answer F1 is used only as a training reward after choosing and executing the action.

Reward: `(F1_after−F1_initial)/100 − .05×(payload/full_broadcast_payload) − .02×(distinct_receiver_calls/3)`. Report full measured inference costs separately; this engineering objective is not a dollar-cost formula or universally optimal weighting. A fan-in invokes one receiver, an out-star two, and a cycle/broadcast three; untouched initial responses must never be counted as new calls.

PPO uses autograd clipping, value regression, entropy, gradient-norm clipping and approximate-KL stopping. Three optimizer seeds share the same authentic frozen initial LLM responses, reducing waste. They are optimizer replicates, not independent backbones or additional independent questions.

After each collection epoch, choose maximum prespecified validation utility, with earlier-epoch tie breaking. The 30-question validation set selects both checkpoint and one of three declared static candidates. Held-out scores cannot affect optimizer updates, early stopping, penalty coefficients, prompts or checkpoint selection. Optimizer, actor/critic and Torch RNG state are saved for resume. A held-out evaluation gate closes further training on that frozen study.

Active sample-size revision: **100 fixed training questions, 30 fixed validation questions, and 300 unchanged separate held-out questions**. Three collection epochs and three optimizer seeds remain. Previous 300/100 artifacts are preserved as a separate study; their predictions and scores did not select the subsets. Reused provider responses retain their original provenance and costs. This notebook has not yet executed the new study to completion.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'protocol.py').exists() else Path.cwd() / 'WEBExtension'
if not (ROOT / 'protocol.py').exists():
    raise FileNotFoundError('Launch Jupyter from the project root or WEBExtension folder')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
from protocol import load_official, write_json, file_hash
from experiments import load_config, preview, freeze, run
CFG = load_config(ROOT / 'config.json')
EXECUTE = False  # Set True only after configuring your real backend and maximum call budget.
print('Notebook root:', ROOT)
print('Inference enabled:', EXECUTE)

SETTINGS=json.loads((ROOT/'controller_config.json').read_text())
from controller_study import forecast, study_directory
STUDY_DIR,SPLIT_DIR=study_directory(CFG,SETTINGS)
print(json.dumps(forecast(CFG,SETTINGS),indent=2))


Notebook root: /home/saurab/Documents/projectRLMAIN/WEBExtension
Inference enabled: False


{
  "study_dir": "/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_study_a0b3768357484b45",
  "split_dir": "/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_splits_2e025f8ae3d9815e",
  "train": 100,
  "validation": 30,
  "heldout": 300,
  "seeds": [
    7,
    19,
    31
  ],
  "conservative_max_calls_before_reuse": 26340,
  "configured_operational_cap": 46000,
  "note": "Upper bound; identical initial and receiver prompts reuse real cached responses. Pilot costs are separate."
}


In [2]:
from controller_study import train
print(train(CFG,SETTINGS,execute=EXECUTE))
status=STUDY_DIR/'training_status.json'
if status.exists():
    print(json.loads(status.read_text()))
else:
    print('No real controller training completed. Unit tests are not learned checkpoints.')

{'study_dir': '/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_study_a0b3768357484b45', 'split_dir': '/home/saurab/Documents/projectRLMAIN/WEBExtension/artifacts/controller_splits_2e025f8ae3d9815e', 'train': 100, 'validation': 30, 'heldout': 300, 'seeds': [7, 19, 31], 'conservative_max_calls_before_reuse': 26340, 'configured_operational_cap': 46000, 'note': 'Upper bound; identical initial and receiver prompts reuse real cached responses. Pilot costs are separate.'}
{'status': 'RUNNING'}


In [3]:
for seed in SETTINGS['training_seeds']:
    path=STUDY_DIR/f'seed_{seed}'/'selection.json'
    print(seed,json.loads(path.read_text()) if path.exists() else 'NO VALIDATION-SELECTED CHECKPOINT')

7 NO VALIDATION-SELECTED CHECKPOINT
19 NO VALIDATION-SELECTED CHECKPOINT
31 NO VALIDATION-SELECTED CHECKPOINT
